# NB-16-MISO · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-16-MISO**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-16-MISO.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This is an observed-data test of identification, rather than a replay of the controlled forecast experiment. It asks whether constraint-dual loadings learned on earlier MISO data remain stable on later data.

- [Volume VI, Chapter 4 — Constraint and topology identification](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c4)
- [Volume VI, Chapter 8 — External validation](https://sherandoresearch.github.io/power-markets-series/vol6_inference_and_decisions.html#c8)
- [Volume III, Chapter 9 — Congestion, prices, and spectral market structure](https://sherandoresearch.github.io/power-markets-series/vol3_spectral_grids.html#c9)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-16-MISO', require_key=True)
print(setup_status)


# NB-16-MISO — Observed congestion identification
Observed-data extension alongside NB-16's product distributions and NB-15's exposure geometry.
**Question:** does an observed constraint dual supply a stable coordinate for
explaining hub congestion spreads? This is an offline, retrospective study of
real MISO day-ahead observations for 1–21 September 2026. No synthetic data is used.

We audit the panel, learn loadings on the first 14 days, assess them on the last
seven, and enforce a training-only stability gate before constructing geometry.
Prerequisites: centered regression, rank-one SVD, and the difference between a
price explanation and a forecast. The retained export and helpers are required
to rerun; this reader and download preserve the executed results.

In [ ]:
from datetime import datetime
import json
import numpy as np
from methods_notebooks.session import Session, setup_plot
session = Session('NB-16-MISO', version='1.43.0')
inputs = session.inputs
nodes = inputs['nodes']
print('Local raw input inspection:',inputs['raw_nodal_rows'][nodes[0]][:1])
m = dict(inputs['observed'])
for field in ('y','centered','x','mask','train','test','h'):
    m[field] = np.asarray(m[field], dtype=bool if field in ('mask','train','test') else float)
m['times'] = [datetime.fromisoformat(t) for t in m['times']]
ridge = {name:session.execute('regression_fit_predict', request)['result']['prediction_result']
    for name,request in inputs['requests'].items() if name.startswith('ridge_')}
pca = session.execute('pca_diagnostics',inputs['requests']['pca'])['result']
assert pca['pca_model_summary']['scaling'] == 'center_only'
assert all(r['selected_alpha'] == 10 and r['alpha_selection'] == 'caller_fixed_single_candidate' for r in ridge.values())
def coefficients(part):
    return np.column_stack([ridge[f'ridge_{part}_{j}']['coefficients_original_units'] for j in range(4)])
m['coef'] = coefficients('full')
m['intercept'] = np.array([ridge[f'ridge_full_{j}']['intercept_original_units'] for j in range(4)])
m['pieces'] = [coefficients('week1'),coefficients('week2')]
pred = np.full((504,4), np.nan)
for j in range(4):
    r = ridge[f'ridge_full_{j}']
    for ids, values in ((r['training_row_ids'],r['training_predictions']),(r['row_ids'],r['predictions'])):
        for row_id,value in zip(ids,values):pred[int(row_id.split(':')[1]),j]=value
diag = pca['reconstruction_diagnostics']
low = np.full((504,4), np.nan)
for ids,values in ((diag['training_row_ids'],diag['training_reconstructed_values']),
                   (diag['evaluation_row_ids'],diag['reconstructed_values'])):
    for row_id,value in zip(ids,values):low[int(row_id.split(':')[1])]=value
assert np.isfinite(low).all() and np.isnan(pred[~m['mask']]).all()
m['singular'] = np.array(pca['pca_model_summary']['training_singular_values'])
m['direct'] = m['y'] @ m['h']
m['low'],m['model'] = low @ m['h'],pred @ m['h']
m['metrics'] = {name:np.sqrt(np.mean((estimate[m['test']]-m['direct'][m['test']])**2,axis=0)).tolist()
    for name,estimate in [('direct identity',m['direct']),('rank-one reconstruction',m['low']),('regularized dual explanation',m['model'])]}
# These are local diagnostics on server-returned fits, not a local estimator.
b1,b2 = m['pieces']
m['delta'] = float(np.linalg.norm(b1-b2)/max(np.linalg.norm(b1),np.linalg.norm(b2),1e-12))
m['cosine'] = float(np.sum(b1*b2)/(np.linalg.norm(b1)*np.linalg.norm(b2)))
m['stable'] = bool(m['delta'] <= .25 and m['cosine'] >= .95)
assert not m['stable'], 'This retained counterexample must continue to fail its declared stability gate'
m['gram'] = None
print(json.dumps(m['audit'],indent=2))
print('Hubs:',', '.join(nodes))

## 1. Time, identity, and absence before regression
The EWB MISO snapshot `miso-20261014T224032Z-62b47895` contains 2,016 prices:
four hubs × 504 hours. Hour ending 1 is 05:00 UTC on the market day, and hour
ending 24 is 04:00 UTC on the next day: MISO's fixed EST clock, without a DST shift.
Each node/time key is unique and every expected hour is present. The selected
names come from the observed node inventory; the broad upstream `Hub` category
also contains other names, so it is not used as our four-hub universe.

All eight constraint-query pages were retained. There are rows in every sampled
hour, but that does **not** prove that an absent individual constraint has zero
shadow price. We use the compound (ID, name, contingency description, constraint
description) identity and keep absent or null duals unknown. No zero filling.
Coverage of returned rows is distinct from completeness of the original reports.

The most frequently observed finite dual in the training window selects one
explanatory constraint, without looking at holdout outcomes. This conservative
one-feature model avoids a mostly missing, wide matrix. Its complete-case sample
is conditional on publication support and is not representative of every hour.

In [ ]:
print('Selected compound identity:',m['key'])
print('Training support:',int(m['train'].sum()),'/ 336; holdout support:',int(m['test'].sum()),'/ 168')
assert np.isnan(m['x'][~m['mask']]).all()
plt=setup_plot()
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
for j,node in enumerate(nodes):axes[0].plot(m['times'],m['centered'][:,j],label=node,linewidth=.8,color=['#88C0D0','#EBCB8B','#A3BE8C','#B48EAD'][j])
axes[0].set(ylabel='Centered MCC · USD/MWh',title='Observed hub congestion prices');axes[0].legend(ncol=2,fontsize=8)
axes[1].plot(m['times'],m['x'][:,0],color='#EBCB8B',linewidth=.9)
axes[1].set(ylabel='Signed dual · USD/MWh',title='Selected constraint: gaps remain unknown',xlabel='UTC')
for ax in axes:ax.axvline(m['times'][336],color='#BF616A',linestyle='--');ax.grid(alpha=.3)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 2. Centered nodal loadings imply consistent path differences
For each hour, subtract the cross-hub MCC mean. Fit `Y = X B + a + residual`
on complete training hours. Ridge uses training-standardized X, an unpenalized
intercept, and a declared penalty of 10; it is not tuned on the holdout. Return
B in original units. Centered targets force both B and a to sum to zero across
nodes, fixing the common reference without identifying physical PTDFs.

Each path has −1 at source and +1 at sink. `Y H` is the **congestion-price
difference** in USD/MWh; for one MW over one hour its numerical value is USD.
It excludes losses, energy components, FTR funding adjustments, and fees. It is
not a claim about the full payout of a particular transmission-right contract.
B H derives paths from one nodal model; independently fitting paths could break
reversal and composition. The negative published duals keep their original signs.

In [ ]:
labels=['AR→IL','IL→MN','AR→IN']
np.testing.assert_allclose(m['coef'].sum(1),0,atol=1e-12)
np.testing.assert_allclose(m['intercept'].sum(),0,atol=1e-12)
np.testing.assert_allclose(m['h'].sum(0),0)
np.testing.assert_allclose(m['centered']@m['h'],m['direct'],atol=1e-12)
np.testing.assert_allclose(m['coef']@(m['h'][:,0]+m['h'][:,1]),(m['coef']@m['h'])[0,0]+(m['coef']@m['h'])[0,1])
print('Nodal loadings (price / signed dual):',m['coef'].round(6).tolist())
print('Path loadings:',(m['coef']@m['h']).round(6).tolist())
print('Centered training singular values:',m['singular'].round(3).tolist())

## 3. Common-support holdout comparison
The direct difference is the observed reference, so its error is zero by identity.
Rank-one SVD learns a subspace from **all 336 training hours**, then projects
contemporaneous holdout hub prices into it. The ridge model uses contemporaneous
realized constraint duals. Both are explanatory reconstructions, not forecasts.
Their RMSEs below use exactly the same 85 complete holdout hours. SVD has access
to more contemporaneous price information, so this is not a fair forecasting
contest. Residuals quantify omitted structure in this sample, not total risk.

Collector receipt occurred after delivery; latest observations cannot establish
what a historical bidder knew. No auction-time backtest or trading claim follows.

In [ ]:
print('Holdout RMSE, USD/MWh; columns:',labels)
for name,values in m['metrics'].items():print(name, np.round(values,3).tolist())
fig,axes=plt.subplots(1,2,figsize=(12,4))
for j,(name,values) in enumerate(list(m['metrics'].items())[1:]):
    axes[0].bar(np.arange(3)+(j-.5)*.35,values,width=.35,label=name,color=['#88C0D0','#B48EAD'][j])
axes[0].set_xticks(range(3),labels);axes[0].set(ylabel='Holdout RMSE · USD/MWh',title='Same observed holdout support');axes[0].legend(fontsize=8)
for key,color in [('low','#88C0D0'),('model','#B48EAD')]:
    residual=(m[key]-m['direct'])[m['test'],0]
    axes[1].plot(np.sort(residual),np.arange(1,len(residual)+1)/len(residual),label={'low':'Rank-one','model':'Dual regression'}[key],color=color)
axes[1].set(xlabel='AR→IL reconstruction residual · USD/MWh',ylabel='Empirical cumulative fraction',title='Residuals remain material');axes[1].legend()
fig.tight_layout();plt.show()

## 4. Stability gate: geometry is withheld
Fit the same selected feature separately on training days 1–7 and 8–14.
Before examining holdout errors, require relative loading change ≤25% and cosine
≥0.95. These are declared teaching thresholds, not externally calibrated tests.
Relative change divides the coefficient-vector difference by the larger norm.
The observed change is about 60%: direction is similar, magnitude is unstable.
Therefore **no weighted geometry is produced**. If the gate passed, a possible
training covariance metric would be `(B H).T Var(X) (B H)`; here it is inadmissible.

The 11 retained element rows share the selected constraint's ID and name, but
their contingency codes do not exactly match the binding report's description.
We do not guess a physical mapping. Subregional balance-hour semantics and the
constraint report's completeness are unresolved, so neither is used to complete
a missing physical model. A scalar observed dual cannot identify a full network;
regularization does not supply that identification.

In [ ]:
print({'relative_loading_change':round(m['delta'],4),'loading_cosine':round(m['cosine'],4),'geometry_gate_passed':m['stable']})
assert not m['stable'] and m['gram'] is None
fig,ax=plt.subplots(figsize=(9,4))
for i,coef in enumerate(m['pieces']):ax.bar(np.arange(4)+(i-.5)*.35,coef[0],width=.35,label='Training week '+str(i+1),color=['#88C0D0','#EBCB8B'][i])
ax.set_xticks(range(4),[n.removesuffix('.HUB') for n in nodes]);ax.set(ylabel='Centered loading · dimensionless',title='Loading magnitude changes across training weeks');ax.legend();fig.tight_layout();plt.show()

## Exercise and next evidence
Why not replace every gap with zero to increase sample size? That would assert
an unverified report-completeness contract and could change both coefficients
and residuals. Why not relax the 25% gate after seeing the result? That changes
the decision rule to accommodate the sample. Preserve the failure instead.
A broader study needs original-report completeness evidence, validated element
mappings, earlier admissible vintages, and repeated out-of-period stability checks.
This notebook supplies an observed identification counterexample, not a substitute
for NB-15's controlled geometry or an official MISO sensitivity matrix.

In [ ]:
import copy
from methods_notebooks.session import MethodsError
negative_controls = []
bad = copy.deepcopy(inputs['requests']['pca'])
bad['inputs']['pca_settings']['scaling'] = 'evaluation_standard_deviation'
negative_controls.append(('pca_diagnostics',bad))
bad = copy.deepcopy(inputs['requests']['pca'])
bad['inputs']['train_eval_panel']['units'][0] = 'MW'
negative_controls.append(('pca_diagnostics',bad))
bad = copy.deepcopy(inputs['requests']['ridge_full_0'])
bad['inputs']['estimator_settings']['alpha_grid'] = [1.0]
negative_controls.append(('regression_fit_predict',bad))
for method,payload in negative_controls:
    try:
        session.execute(method,payload)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected unsupported-profile rejection:',method,str(exc))
    else:
        raise AssertionError('Unsupported profile accepted')
(session.output/'consumer-summary.json').write_text(json.dumps(dict(
    coef=m['coef'].tolist(),intercept=m['intercept'].tolist(),pieces=[p.tolist() for p in m['pieces']],
    metrics=m['metrics'],delta=m['delta'],cosine=m['cosine'],stable=m['stable'],gram=m['gram'],
    singular=m['singular'].tolist(),low=m['low'].tolist(),
    model=np.where(np.isnan(m['model']),None,m['model']).tolist()),indent=2,allow_nan=False)+'\n')
summary=session.finish({'observed_only':True,'complete_hub_panel':True,'unknown_duals_preserved':True,
    'training_only_selection':True,'geometry_withheld_on_instability':True,
    'no_physical_mapping_inferred':True,'new_synthetic_data':False},
    {'ridge_penalty':10,'training_days':14,'holdout_days':7,'stability_relative_change_max':.25,'stability_cosine_min':.95})